# Module 7: Advanced Consumer

本章目标：
- 掌握手动 offset 提交，实现精准的消费进度控制
- 使用 `seek` 从指定位置重放消息
- 实现「至少一次」（At-Least-Once）处理语义
- 结合事务 Producer 实现「精确一次」（Exactly-Once）语义
- 理解 Consumer 的 `poll` 超时与心跳机制

---

## 前置条件

- 集群运行中：`docker compose up -d`
- 已安装依赖：`uv sync`

## 消费语义对比

| 语义 | 实现方式 | 业务影响 |
|------|---------|----------|
| At-Most-Once | 先提交 offset，再处理 | 崩溃时消息可能丢失 |
| At-Least-Once | 先处理成功，再提交 offset | 崩溃时可能重复处理 |
| Exactly-Once | 事务 + 原子提交 offset | 不丢不重，但开销最高 |

In [ ]:
import asyncio
from aiokafka import AIOKafkaProducer, AIOKafkaConsumer, TopicPartition
from aiokafka.admin import AIOKafkaAdminClient, NewTopic

BROKERS = "localhost:19094,localhost:29094,localhost:39094"
TOPIC = "advanced-consumer-demo"

async def setup(brokers, topic, count=20):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        if topic not in existing:
            await admin.create_topics([
                NewTopic(name=topic, num_partitions=1, replication_factor=3)
            ])
    finally:
        await admin.close()

    producer = AIOKafkaProducer(bootstrap_servers=brokers)
    await producer.start()
    try:
        for i in range(count):
            await producer.send_and_wait(topic, f"record-{i:03d}".encode())
        print(f"✓ Seeded {count} records into '{topic}'")
    finally:
        await producer.stop()

await setup(BROKERS, TOPIC)

## 1. 手动提交 offset（At-Least-Once）

关闭 `enable_auto_commit`，处理完成后手动调用 `commit()`。
如果在处理期间崩溃，下次重启从上次成功 commit 的 offset 开始，消息会被重复处理。

In [ ]:
async def manual_commit_demo(brokers, topic, group="manual-commit-group"):
    consumer = AIOKafkaConsumer(
        topic,
        bootstrap_servers=brokers,
        group_id=group,
        auto_offset_reset="earliest",
        enable_auto_commit=False,
    )
    await consumer.start()
    total = 0
    try:
        while True:
            results = await consumer.getmany(timeout_ms=1500, max_records=5)
            if not results:
                break  # 无新消息，退出
            batch = [msg for msgs in results.values() for msg in msgs]
            values = [m.value.decode() for m in batch]
            print(f"  Processing batch: {values}")
            await consumer.commit()
            print(f"  ✓ Committed offset up to {batch[-1].offset}")
            total += len(batch)
    finally:
        await consumer.stop()
    print(f"  Total processed: {total} messages")

await manual_commit_demo(BROKERS, TOPIC)

## 2. Seek：从指定 offset 重放消息

典型场景：
- 数据处理 Bug 修复后，需要从出错点重新处理
- 回放最近 N 小时的数据做分析

In [ ]:
async def seek_demo(brokers, topic, seek_to_offset=10):
    consumer = AIOKafkaConsumer(
        bootstrap_servers=brokers,
        auto_offset_reset="earliest",
    )
    tp = TopicPartition(topic, 0)
    consumer.assign([tp])
    await consumer.start()
    try:
        consumer.seek(tp, seek_to_offset)
        print(f"  Seeked to offset={seek_to_offset}, reading next 5 messages:")
        results = await consumer.getmany(tp, timeout_ms=2000, max_records=5)
        for msg in results.get(tp, []):
            print(f"  offset={msg.offset}: {msg.value.decode()}")
    finally:
        await consumer.stop()

print("=== Seek to offset 10 ===")
await seek_demo(BROKERS, TOPIC, seek_to_offset=10)

In [ ]:
# Seek to beginning / end — query offset metadata
async def seek_to_end_demo(brokers, topic):
    consumer = AIOKafkaConsumer(
        bootstrap_servers=brokers,
    )
    tp = TopicPartition(topic, 0)
    consumer.assign([tp])
    await consumer.start()
    try:
        await consumer.seek_to_end(tp)
        pos = await consumer.position(tp)
        end_offsets = await consumer.end_offsets([tp])
        beginning_offsets = await consumer.beginning_offsets([tp])
        print(f"  Partition 0:")
        print(f"    beginning offset = {beginning_offsets[tp]}")
        print(f"    end offset       = {end_offsets[tp]}")
        print(f"    total messages   = {end_offsets[tp] - beginning_offsets[tp]}")
    finally:
        await consumer.stop()

print("=== Partition offset info ===")
await seek_to_end_demo(BROKERS, TOPIC)

## 3. 基于时间戳的 Seek

回放某个时间点之后的消息，不需要知道具体 offset。

In [ ]:
import time

async def seek_by_timestamp(brokers, topic, seconds_ago=60):
    """读取最近 N 秒内的消息。"""
    target_ts = int((time.time() - seconds_ago) * 1000)

    consumer = AIOKafkaConsumer(bootstrap_servers=brokers)
    tp = TopicPartition(topic, 0)
    consumer.assign([tp])
    await consumer.start()
    try:
        offsets_for_times = await consumer.offsets_for_times({tp: target_ts})
        offset_and_ts = offsets_for_times[tp]
        if offset_and_ts is None:
            print(f"  No messages after {seconds_ago}s ago")
            return

        print(f"  Seeking to offset={offset_and_ts.offset} (messages after {seconds_ago}s ago)")
        consumer.seek(tp, offset_and_ts.offset)

        all_msgs = []
        while True:
            results = await consumer.getmany(tp, timeout_ms=1000, max_records=100)
            if not results or not results.get(tp):
                break
            all_msgs.extend(results[tp])

        print(f"  Total: {len(all_msgs)} messages in the last {seconds_ago}s")
        for msg in all_msgs[:5]:
            print(f"  offset={msg.offset}: {msg.value.decode()}")
        if len(all_msgs) > 5:
            print(f"  ... ({len(all_msgs)} total)")
    finally:
        await consumer.stop()

print("=== Seek by timestamp (last 5 minutes) ===")
await seek_by_timestamp(BROKERS, TOPIC, seconds_ago=300)

## 4. 精确一次语义（Exactly-Once）

结合事务 Producer，将「消费进度提交」和「处理结果写入」合并到同一事务：
- 两者要么都成功，要么都回滚
- 消费者端需设置 `isolation_level="read_committed"` 只读可见的提交消息

In [ ]:
OUTPUT_TOPIC = "eos-output"

async def setup_eos_topics(brokers):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        if OUTPUT_TOPIC not in existing:
            await admin.create_topics([
                NewTopic(name=OUTPUT_TOPIC, num_partitions=1, replication_factor=3)
            ])
            print(f"✓ Created {OUTPUT_TOPIC}")
    finally:
        await admin.close()

async def exactly_once_processor(brokers, input_topic, output_topic, group, count=5):
    """Read-Process-Write with exactly-once semantics."""
    consumer = AIOKafkaConsumer(
        input_topic,
        bootstrap_servers=brokers,
        group_id=group,
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        isolation_level="read_committed",  # 只读已提交事务的消息
    )
    producer = AIOKafkaProducer(
        bootstrap_servers=brokers,
        transactional_id=f"{group}-tx",
        enable_idempotence=True,
        acks="all",
    )
    await consumer.start()
    await producer.start()
    processed = 0
    try:
        async for msg in consumer:
            # 处理消息
            result = msg.value.decode().upper()  # 示例：转大写

            # 原子事务：写结果 + 提交 offset
            await producer.begin_transaction()
            try:
                await producer.send(output_topic, result.encode())
                await producer.send_offsets_to_transaction(
                    {TopicPartition(input_topic, msg.partition): msg.offset + 1},
                    group,
                )
                await producer.commit_transaction()
                print(f"  EOS: {msg.value.decode()!r} → {result!r} (offset={msg.offset} committed)")
            except Exception as e:
                await producer.abort_transaction()
                print(f"  Aborted: {e}")
                raise

            processed += 1
            if processed >= count:
                break
    finally:
        await consumer.stop()
        await producer.stop()
    print(f"  Processed {processed} messages with exactly-once semantics")

await setup_eos_topics(BROKERS)
print("=== Exactly-Once Processing ===")
await exactly_once_processor(BROKERS, TOPIC, OUTPUT_TOPIC, "eos-group", count=5)

## 总结

| 操作 | API | 说明 |
|------|-----|------|
| 手动提交 | `consumer.commit()` | 精确控制消费进度 |
| Seek to offset | `consumer.seek(tp, offset)` | 重放指定位置 |
| Seek to timestamp | `consumer.offsets_for_times()` + `seek()` | 按时间重放 |
| Seek to end | `consumer.seek_to_end(tp)` | 只消费最新消息 |
| Exactly-Once | 事务 + `send_offsets_to_transaction` | 不丢不重 |

## 下一章

**Module 8: Taskiq Integration** — 用 Kafka 作为消息后端构建分布式异步任务队列。